# Reproduce the analysis

Replication walkthrough for **"The Informativeness of Frequency-Report Scoring Rules"** by Jesper Armouti-Hansen.

This notebook walks from a worked example to the paper's headline tables. The parameter cell below is the only place numbers live; the rest of the notebook reads from it, so you can change `N`, `K`, `ALPHA` and re-run to explore.

- **Setup** (cell below) imports the math library and sets parameters.
- **A worked example** illustrates the three rules on a small instance.
- **A small simulation** demonstrates the regime crossover with 500 draws (the paper uses 5,000).
- **The committed CSVs** reproduce the paper's Table 1 (win shares), Table 2 (regret), and the asymmetric-Dirichlet robustness table from Appendix C.

The full paper simulation (`uv run python scripts/design_efficiency.py --final --draws 5000`) takes about 30 minutes; this notebook never reruns it.

In [1]:
# Setup --- imports and parameters.
import sys
from pathlib import Path

# Make scripts/ importable.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import numpy as np
import pandas as pd

# Parameters --- change these and rerun the simulation cell below to explore.
N = 20                              # number of trials
K = 5                               # number of categories
ALPHA_VALUES = [0.1, 0.3, 1.0, 3.0] # symmetric Dirichlet concentrations to sweep
N_DRAWS = 500                       # belief draws per alpha (paper uses 5,000)
SEED = 20260508

assert K >= 2, "K = 1 is degenerate (single category, no inference)."
print(f"ROOT = {ROOT}")


ROOT = /Users/jesper/repos/frequency-beliefs


## What the analysis computes

The paper studies *belief elicitation with frequency reports*. A subject reports a count vector $r = (r_1, \dots, r_k)$ with $\sum_i r_i = n$, summarising their belief about how the $n$ outcomes will split across $k$ categories. A scoring rule pays the subject based on the distance between $r$ and the realised count vector $\omega$.

Three rules are compared:

| Internal name | Paper name | Penalty $D(r, \omega)$ |
|---|---|---|
| `discrete` | Frequency-guessing | $\mathbf{1}\{r \neq \omega\}$ (fixed prize on exact match) |
| `quadratic` | Squared-distance | $\sum_i (r_i - \omega_i)^2$ |
| `manhattan` | Manhattan distance | $\sum_i \lvert r_i - \omega_i \rvert$ |

Each rule induces an *identified set* $P_S(r) = \{p : r \text{ is optimal under } p\}$. The width of $P_S(r)$ in each coordinate measures how informative the report is.

In [2]:
# A worked example. p = (0.5, 0.3, 0.2), n = 3, k = 3.
import numpy as np
from design_efficiency import report_for_rule, manhattan_sharp_coordinate_bounds
from verify_regions import (
    exact_coordinate_intervals,
    squared_closed_form_bounds,
)

p = np.array([0.5, 0.3, 0.2])
n_ex, k_ex = 3, 3

# Optimal report under each rule, using the same tie-breaking the simulation uses.
r_freq, _ = report_for_rule("discrete", p, n_ex)
r_squared, _ = report_for_rule("quadratic", p, n_ex)
r_manhattan, _ = report_for_rule("manhattan", p, n_ex)

# Sharp coordinate bounds. For Manhattan we use the sharp root-find variant the
# paper's tables rely on (not the threshold-grid approximation).
bounds_freq = exact_coordinate_intervals(r_freq, n_ex, k_ex)
bounds_squared = squared_closed_form_bounds(r_squared, n_ex, k_ex)
bounds_manhattan = manhattan_sharp_coordinate_bounds(r_manhattan, n_ex, k_ex)

def fmt(b):
    return f"[{b[0]:.3f}, {b[1]:.3f}]"

rows = [
    {"Rule": "Frequency-guessing", "Optimal report": r_freq, "p1": fmt(bounds_freq[0]), "p2": fmt(bounds_freq[1]), "p3": fmt(bounds_freq[2])},
    {"Rule": "Squared-distance", "Optimal report": r_squared, "p1": fmt(bounds_squared[0]), "p2": fmt(bounds_squared[1]), "p3": fmt(bounds_squared[2])},
    {"Rule": "Manhattan", "Optimal report": r_manhattan, "p1": fmt(bounds_manhattan[0]), "p2": fmt(bounds_manhattan[1]), "p3": fmt(bounds_manhattan[2])},
]
print(f"True belief p = {p.tolist()}, n = {n_ex}, k = {k_ex}\n")
pd.DataFrame(rows)


True belief p = [0.5, 0.3, 0.2], n = 3, k = 3



,Rule,Optimal report,p1,p2,p3
0,Frequency-guessing,"(2, 1, 0)","[0.400, 0.750]","[0.200, 0.500]","[0.000, 0.250]"
1,Squared-distance,"(1, 1, 1)","[0.111, 0.556]","[0.111, 0.556]","[0.111, 0.556]"
2,Manhattan,"(2, 1, 0)","[0.425, 0.794]","[0.142, 0.500]","[0.000, 0.233]"


Each row shows the report a subject with belief `p` would submit under that rule, and the coordinate-wise identification interval the researcher recovers from observing that report. The true `p` lies inside each interval --- that is the partial-identification guarantee.

## Running a small simulation

The paper compares the three rules across a grid of $(n, k, \alpha)$ values, where $\alpha$ controls how concentrated the latent beliefs are. The cell below draws `N_DRAWS = 500` beliefs from a symmetric $\mathrm{Dirichlet}(\alpha)$ at the parameters set at the top, computes each rule's optimal report and bounds, and reports the *win share* --- the fraction of draws on which each rule gives the narrowest average coordinate interval.

Change `ALPHA` in the setup cell to see how the ranking shifts: small `alpha` favours squared-distance, large `alpha` favours frequency-guessing, with the crossover near `alpha = 1`.

In [3]:
from design_efficiency import run_latent_simulation

# Conditional on optimal reporting --- the simulation assumes subjects report
# the optimal count vector for their belief under each rule. Whether real
# subjects do so is a separate empirical question (see paper Section 7).

aggregate_rows, win_rows = run_latent_simulation(
    n_values=[N],
    k_values=[K],
    alpha_values=ALPHA_VALUES,
    draws=N_DRAWS,
    seed=SEED,
    tolerance=1e-4,
)

label_to_paper = {
    "Discrete metric": "Frequency-guessing",
    "Quadratic distance": "Squared-distance",
    "Manhattan distance": "Manhattan",
}
paper_order = ["Frequency-guessing", "Squared-distance", "Manhattan"]

df_wins = pd.DataFrame(win_rows)
df_avg = (
    df_wins[df_wins["metric"] == "coord_avg"]
    .assign(rule=lambda d: d["rule"].map(label_to_paper))
    .pivot(index="alpha", columns="rule", values="win_share")
    .reindex(columns=paper_order)
    .round(2)
)
print(f"Win shares at n={N}, k={K}, draws={N_DRAWS} per alpha")
print("(small alpha => sparse beliefs, large alpha => balanced; crossover near alpha=1)\n")
df_avg


Win shares at n=20, k=5, draws=500 per alpha
(small alpha => sparse beliefs, large alpha => balanced; crossover near alpha=1)



rule,Frequency-guessing,Squared-distance,Manhattan
alpha,,,
0.1,0.00,0.91,0.09
0.3,0.06,0.76,0.19
1.0,0.52,0.34,0.14
3.0,0.96,0.02,0.01


## Reproducing the paper's headline tables

The next three cells read the committed CSVs in `outputs/design_exercise/`, which are from the full `--final --draws 5000` simulation. No new simulation is run.

### Table 1 --- win shares by belief concentration

In [4]:
df = pd.read_csv(ROOT / "outputs/design_exercise/rule_comparison.csv")

# Average across the (n, k) grid at each alpha, for the two headline metrics.
table1 = (
    df[df["metric"].isin(["coord_avg", "mean_linear"])]
    .assign(rule=lambda d: d["rule"].map(label_to_paper))
    .groupby(["metric", "alpha", "rule"])["win_share"].mean()
    .unstack("rule")
    .reindex(columns=paper_order)
    .round(2)
)
table1


rule               Frequency-guessing  Squared-distance  Manhattan
metric      alpha                                                 
coord_avg   0.1                  0.08              0.84       0.07
            0.3                  0.28              0.58       0.14
            1.0                  0.76              0.16       0.08
            3.0                  0.98              0.01       0.01
            10.0                 1.00              0.00       0.00
mean_linear 0.1                  0.08              0.83       0.10
            0.3                  0.22              0.60       0.18
            1.0                  0.58              0.27       0.15
            3.0                  0.83              0.13       0.04
            10.0                 0.92              0.06       0.01

### Table 2 --- mean regret by belief concentration

Regret is the rule's interval width minus the smallest width among the three rules; lower is better. Frequency-guessing and squared-distance each have a regime with near-zero regret and a regime with substantial regret; Manhattan's regret is nearly flat across regimes --- the regime-robust choice.

In [5]:
table2 = (
    df[df["metric"].isin(["coord_avg", "mean_linear"])]
    .assign(rule=lambda d: d["rule"].map(label_to_paper))
    .groupby(["metric", "alpha", "rule"])["mean_regret"].mean()
    .unstack("rule")
    .reindex(columns=paper_order)
    .round(3)
)
table2


rule               Frequency-guessing  Squared-distance  Manhattan
metric      alpha                                                 
coord_avg   0.1                 0.032             0.002      0.011
            0.3                 0.016             0.005      0.007
            1.0                 0.003             0.012      0.007
            3.0                 0.000             0.019      0.011
            10.0                0.000             0.023      0.014
mean_linear 0.1                 0.047             0.002      0.013
            0.3                 0.024             0.005      0.009
            1.0                 0.006             0.013      0.008
            3.0                 0.001             0.020      0.012
            10.0                0.000             0.024      0.016

### Asymmetric-Dirichlet robustness (Appendix C)

Nine asymmetric concentration vectors at `n = 20, k = 5`. The symmetric Dirichlet cannot reproduce these geometries; the robustness check confirms the regime story carries over, with one exception (squared-distance extends its dominance into the high-spread balanced cells of the linear-mean target).

In [6]:
df_robust = pd.read_csv(ROOT / "outputs/design_exercise/robustness/rule_comparison.csv")

# Both panels: average-coordinate and linear-mean targets.
table_robust = (
    df_robust[df_robust["metric"].isin(["coord_avg", "mean_linear"])]
    .assign(rule=lambda d: d["rule"].map(label_to_paper))
    .set_index(["metric", "alpha_label", "rule"])["win_share"]
    .unstack("rule")
    .reindex(columns=paper_order)
    .round(2)
)
table_robust


rule                          Frequency-guessing  Squared-distance  Manhattan
metric      alpha_label                                                      
coord_avg   balanced-extreme                0.53              0.32       0.15
            balanced-graded                 0.76              0.17       0.07
            balanced-mild                   0.87              0.08       0.05
            balanced-skewed                 0.84              0.10       0.06
            balanced-steep                  0.50              0.33       0.17
            graded                          0.07              0.67       0.26
            one-dominant                    0.06              0.60       0.33
            transition                      0.26              0.54       0.20
            two-modes                       0.03              0.76       0.21
mean_linear balanced-extreme                0.05              0.82       0.14
            balanced-graded                 0.52              0.41       0.07
            balanced-mild                   0.82              0.14       0.04
            balanced-skewed                 0.68              0.26       0.06
            balanced-steep                  0.08              0.77       0.15
            graded                          0.03              0.86       0.11
            one-dominant                    0.04              0.72       0.24
            transition                      0.23              0.63       0.15
            two-modes                       0.04              0.82       0.14

## Modifying the analysis

Change `N`, `K`, `ALPHA`, or `N_DRAWS` in the setup cell and re-run the *small simulation* cell to see your own numbers. The headline-table cells read the committed CSVs, so they are tied to the paper's parameter grid; to regenerate those CSVs at different parameters, run

```bash
uv run python scripts/design_efficiency.py --final --draws 5000
```

from the repository root (about 30 minutes).

For an exhaustive characterisation of the math --- the closed-form bounds, the worked-example tables, and the structural lemma --- see the paper at `paper/main.pdf` and the proofs in `paper/sections/08_appendix.tex`.